# **Notebook : Post-R Visualization (Figures 4 & 5)**

Ce notebook génère les figures finales après l'analyse R (Limma/Voom + GO enrichment).

**Prérequis** :
- Résultats Limma exportés en CSV : `limma_results_AD_vs_CTRL.csv`, `limma_results_PD_vs_CTRL.csv`
- Résultats GO exportés en CSV : `ORA_results_AD.csv`, `ORA_results_PD.csv`

**Sections** :
1. Setup
2. Figure 4 : Volcano Plot (AD + PD)
3. Figure 5 : ORA Barplot (Top 10 GO Terms)

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Définir les dossiers (adapter selon ton projet)
PROJECT_ROOT = "C:/Z/AIDA_transcriptomics_project/transcriptomics-code"  # À ADAPTER

DIRS = {
    'DATA': os.path.join(os.path.dirname(PROJECT_ROOT), 'data'),
    'FIGURES': os.path.join(PROJECT_ROOT, 'figures')
}

print(f"✅ Dossiers configurés")
print(f"   DATA: {DIRS['DATA']}")
print(f"   FIGURES: {DIRS['FIGURES']}")


# **Figure 4 : Volcano Plot Stratégique**

Visualisation des gènes différentiellement exprimés (DEGs) pour AD et PD.

**Seuils** :
- |log2FC| > 0.5 (fold-change 1.41x)
- Adjusted p-value < 0.05

**Output** : `Fig4_Volcano_Strategic.png`

In [ ]:
# --- FIGURE 4 : VOLCANO PLOT ---
print("="*70)
print("🌋 Génération Figure 4 : Volcano Plot")
print("="*70)

# Chargement des résultats Limma
results_ad = pd.read_csv(os.path.join(DIRS['DATA'], 'limma_results_AD_vs_CTRL.csv'))
results_pd = pd.read_csv(os.path.join(DIRS['DATA'], 'limma_results_PD_vs_CTRL.csv'))

def plot_volcano(df, title, ax):
    df = df.copy()
    df['color'] = 'gray'
    df.loc[(df['logFC'] > 0.5) & (df['adj.P.Val'] < 0.05), 'color'] = 'red'
    df.loc[(df['logFC'] < -0.5) & (df['adj.P.Val'] < 0.05), 'color'] = 'blue'
    df['-log10p'] = -np.log10(df['adj.P.Val'].clip(lower=1e-300))
    
    for color, label in [('gray','NS'), ('blue','Down'), ('red','Up')]:
        subset = df[df['color']==color]
        ax.scatter(subset['logFC'], subset['-log10p'], c=color, s=10, alpha=0.6, label=f"{label} ({len(subset)})")
    
    ax.axhline(-np.log10(0.05), color='black', linestyle='--', lw=0.8, alpha=0.5)
    ax.axvline(0.5, color='black', linestyle='--', lw=0.8, alpha=0.5)
    ax.axvline(-0.5, color='black', linestyle='--', lw=0.8, alpha=0.5)
    
    # Top genes
    top = df.nsmallest(10, 'adj.P.Val')
    for _, row in top.iterrows():
        if abs(row['logFC']) > 0.5:
            ax.text(row['logFC'], row['-log10p'], row['gene_symbol'], fontsize=7, alpha=0.8)
    
    ax.set_xlabel('log2(Fold Change)', fontsize=12)
    ax.set_ylabel('-log10(Adjusted P-value)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_volcano(results_ad, 'AD vs CTRL', axes[0])
plot_volcano(results_pd, 'PD vs CTRL', axes[1])
plt.tight_layout()
plt.savefig(os.path.join(DIRS['FIGURES'], 'Fig4_Volcano_Strategic.png'), dpi=300, bbox_inches='tight')
print("✅ Sauvegardée : Fig4_Volcano_Strategic.png")
plt.show()
print("="*70)


# **Figure 5 : ORA Barplot (Top 10 GO Terms)**

Visualisation des processus biologiques enrichis.

**Output** : `Fig5_ORA_Barplot.png`

In [ ]:
# --- FIGURE 5 : ORA BARPLOT ---
print("="*70)
print("📊 Génération Figure 5 : ORA Barplot")
print("="*70)

# Chargement des résultats GO
ora_ad = pd.read_csv(os.path.join(DIRS['DATA'], 'ORA_results_AD.csv'))
ora_pd = pd.read_csv(os.path.join(DIRS['DATA'], 'ORA_results_PD.csv'))

def plot_ora(df, title, ax):
    df_top = df.nsmallest(10, 'P.value').copy()
    df_top['-log10p'] = -np.log10(df_top['P.value'].clip(lower=1e-300))
    
    if 'GeneRatio' in df_top.columns:
        def parse_ratio(x):
            if isinstance(x, str) and '/' in x:
                n, d = x.split('/')
                return float(n)/float(d)
            return 0.1
        df_top['ratio'] = df_top['GeneRatio'].apply(parse_ratio)
    else:
        df_top['ratio'] = 0.1
    
    df_top = df_top.sort_values('-log10p')
    ax.barh(range(len(df_top)), df_top['-log10p'], color=plt.cm.Reds(df_top['ratio']))
    ax.set_yticks(range(len(df_top)))
    ax.set_yticklabels(df_top['Term'], fontsize=10)
    ax.set_xlabel('-log10(P-value)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axvline(-np.log10(0.05), color='black', linestyle='--', lw=1, alpha=0.5)
    ax.grid(axis='x', alpha=0.3)
    
    sm = plt.cm.ScalarMappable(cmap='Reds', norm=plt.Normalize(0, 0.5))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Gene Ratio', fontsize=10)

fig, axes = plt.subplots(2, 1, figsize=(12, 10))
plot_ora(ora_ad, 'Top 10 GO Terms: AD vs CTRL', axes[0])
plot_ora(ora_pd, 'Top 10 GO Terms: PD vs CTRL', axes[1])
plt.tight_layout()
plt.savefig(os.path.join(DIRS['FIGURES'], 'Fig5_ORA_Barplot.png'), dpi=300, bbox_inches='tight')
print("✅ Sauvegardée : Fig5_ORA_Barplot.png")
plt.show()
print("="*70)


# **Récapitulatif**

✅ Figure 4 : `Fig4_Volcano_Strategic.png`

✅ Figure 5 : `Fig5_ORA_Barplot.png`

Les figures sont sauvegardées dans `DIRS['FIGURES']`.